# Strategy 1 vs Strategy 2 Comparison

This notebook compares the MKID order-sorting strategies across representative source classes and energy resolutions.

It focuses on the question from the project brief: when does probabilistic assignment outperform grey-zone hard cuts, and where does discarding ambiguous photons become preferable?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from mkid_ifts_sim import InstrumentConfig, Spectrum, load_template, snr_from_time
from mkid_ifts_sim.source import normalize_to_ab_magnitude


def build_sources(config: InstrumentConfig) -> dict[str, Spectrum]:
    sigma = config.sigma_grid()
    return {
        "HII region": load_template("hii_region", sigma_grid_cm=sigma, line_flux=2.0e-2),
        "Faint galaxy": load_template("composite_galaxy", sigma_grid_cm=sigma, magnitude=22.0, band="r"),
        "Planetary nebula": load_template("planetary_nebula", sigma_grid_cm=sigma),
    }


def baseline_config() -> InstrumentConfig:
    return InstrumentConfig(
        n_steps=512,
        n_sigma=2048,
        delta_x_m=2.0e-6,
        t_exp_per_step_s=2.0,
        moon_phase="new",
        apodization="none",
        phase_method="mertz",
        k_sigma=2.0,
    )

In [ ]:
energy_resolutions = [20.0, 35.0, 50.0, 80.0]
strategies = ["hard_cut", "probabilistic"]
config = baseline_config()
sources = build_sources(config)
ref_wavelengths_nm = {
    "HII region": 656.3,
    "Faint galaxy": 750.0,
    "Planetary nebula": 500.7,
}


def evaluate_source(source: Spectrum, source_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    median_snr = np.zeros((len(energy_resolutions), len(strategies)))
    ref_snr = np.zeros_like(median_snr)
    ratio_curves = []

    hard_curves = []
    prob_curves = []
    wavelength_nm = None
    for i, r_energy in enumerate(energy_resolutions):
        curves = {}
        for j, strategy in enumerate(strategies):
            cfg = config.with_updates(R_energy_ref=r_energy, strategy=strategy)
            result = snr_from_time(source, cfg, cfg.total_observing_time_s)
            median_snr[i, j] = np.median(result.snr)
            ref_snr[i, j] = np.interp(
                ref_wavelengths_nm[source_name],
                result.wavelength_nm[::-1],
                result.snr[::-1],
            )
            curves[strategy] = result.snr[::-1]
            wavelength_nm = result.wavelength_nm[::-1]
        hard_curves.append(curves["hard_cut"])
        prob_curves.append(curves["probabilistic"])
        ratio_curves.append(np.divide(curves["probabilistic"], curves["hard_cut"], out=np.ones_like(curves["hard_cut"]), where=curves["hard_cut"] > 0))
    return wavelength_nm, ref_snr, np.asarray(ratio_curves)


results = {name: evaluate_source(source, name) for name, source in sources.items()}

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(14, 4 * len(results)), constrained_layout=True)

for row, (name, (wavelength_nm, ref_snr, ratio_curves)) in enumerate(results.items()):
    ax_heat = axes[row, 0]
    ax_ratio = axes[row, 1]

    image = ax_heat.imshow(
        ref_snr,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[-0.5, 1.5, energy_resolutions[0], energy_resolutions[-1]],
    )
    ax_heat.set_title(f"{name}: SNR at reference wavelength")
    ax_heat.set_xticks([0, 1], ["hard_cut", "probabilistic"])
    ax_heat.set_ylabel("R_E at 500 nm")
    fig.colorbar(image, ax=ax_heat, label="SNR")

    for r_energy, ratio in zip(energy_resolutions, ratio_curves):
        ax_ratio.plot(wavelength_nm, ratio, label=f"R_E={r_energy:.0f}")
    ax_ratio.axhline(1.0, color="black", linestyle="--", linewidth=1)
    ax_ratio.set_title(f"{name}: probabilistic / hard-cut")
    ax_ratio.set_xlabel("Wavelength [nm]")
    ax_ratio.set_ylabel("SNR ratio")
    ax_ratio.legend()

plt.show()